# Accelerate Large Linear Programs using Distributed PDLP on Multi GPU

NVIDIA cuOpt solves very large Linear Programs (LP) very efficiently on GPU with its implementation of the first-order optimization method known as PDLP (primal-dual hybrid gradient method for LP). This technique can now be run on multi-GPU to further speed up the solution process and allow solving existing LPs faster or solving very large LPs that do not fit in a single GPU memory.

<b>Prerequisite</b>: to run cuOpt PDLP solver on multi-GPU, the node where this notebook is running must have >1 gpus available.

Let's start by verifying your Python version. If cuOpt is not already installed, uncomment the cuOpt lines in the next cell.

In [ ]:
# Skip notebook test
import sys
current_version = sys.version_info[:2]
if (3, 11) <= current_version <= (3, 14):
    print(f"✅ Success! Your Python version ({sys.version.split()[0]}) is supported.")
else:
    print(f"❌ Version error! Expected >= 3.11 and <= 3.14. Found: {sys.version.split()[0]}")

In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#%pip uninstall -y cuda-python cuda-bindings cuda-core
# For CUDA 12:
#%pip install --upgrade --only-binary=:all: cuopt-cu12
# For CUDA 13:
#%pip install --upgrade --only-binary=:all: cuopt-cu13
%pip install -q matplotlib

Let's get the path to the cuOpt binary

In [ ]:
import os
from importlib.metadata import distributions

candidates = (
    package.locate_file(entry).resolve()
    for package in distributions()
    if "cuopt" in package.metadata.get("Name", "").lower()
    for entry in package.files or []
    if entry.name == "cuopt_cli"
)

cli_path = next(
    (
        path for path in candidates
        if path.is_file() and os.access(path, os.X_OK)
    ),
    None,
)

if cli_path is None:
    raise FileNotFoundError(
        "No executable cuopt_cli was found in this kernel's installed packages. "
        "If cuOpt is not installed yet (for example in Google Colab), uncomment the "
        "cuOpt install lines in the install cell above, run it, restart the kernel, "
        "and run this cell again."
    )

CUOPT_CLI = str(cli_path)
CUOPT_HOME = str(cli_path.parent)
os.environ["CUOPT_CLI"] = CUOPT_CLI
os.environ["CUOPT_HOME"] = CUOPT_HOME

print(f"Using: {CUOPT_CLI}")

Let's download a small LP

In [ ]:
!wget https://raw.githubusercontent.com/coin-or-tools/Data-Sample/refs/heads/master/afiro.mps

Now let's ensure we can solve a basic instance (`afiro`) using the **PDLP** agorithm (by setting `method 1`). It is this algorithm (PDLP) that is now accelerated on multiple GPUs.

In [ ]:
!$CUOPT_HOME/cuopt_cli --method=1 afiro.mps

We now build a simple helper to run the solver, parse the log and return the time result.

In [ ]:
import re
import subprocess
import os 
# Each successful solve appends {"instance", "num_gpus", "time_s"}.
solve_times = []


def run_cuopt(mps_path, *extra_args, num_gpus=None):
    cmd = [
        os.environ['CUOPT_HOME']+"/cuopt_cli", mps_path, "--method", "1",
        "--relative-primal-tolerance", "1e-6",
        "--relative-dual-tolerance", "1e-6",
        "--relative-gap-tolerance", "1e-6",
        *extra_args,
    ]

    if num_gpus is not None:
        cmd += ["--num-gpus", str(num_gpus)]

    print(
        f"\nRunning {mps_path} with "
        f"{num_gpus if num_gpus is not None else 1} GPUs\n"
    )

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    log_lines = []

    for line in process.stdout:
        print(line, end="", flush=True)
        log_lines.append(line)

    process.wait()
    log = "".join(log_lines)

    match = re.search(r"^Status:.*Time:\s*([0-9.]+)s", log, re.M)

    if match is None:
        raise RuntimeError(
            f"No solver Time in cuopt_cli output (exit code {process.returncode})"
        )

    time_s = float(match.group(1))

    solve_times.append({
        "instance": mps_path.rsplit("/", 1)[-1],
        "num_gpus": num_gpus,
        "time_s": time_s,
    })

    return time_s


In [ ]:
t_afiro = run_cuopt("afiro.mps")

In [ ]:
print(f"Extracted afiro solve time: {t_afiro}s")

## Solving with Multi-GPU PDLP

Let's try again to solve `afiro_original`, but this time with 4 gpus, using the parameter `--num-gpus 4`

In [ ]:
t_afiro_4_GPU = run_cuopt("afiro.mps", num_gpus=4)

print(f"\n\nExtracted afiro solve time: {t_afiro_4_GPU}")

In [ ]:
print(f"1 GPU : {t_afiro}s")
print(f"4 GPU : {t_afiro_4_GPU}s")

We don't see much speedup between 1 GPU solve and 4 GPU solve on `afiro`. This is due to the small size of this instance.

Now we download zib03, a large-scale LP problem 

In [ ]:
!wget -P datasets/linear_programming/  https://miplib2010.zib.de/contrib/miplib2003-contrib/IPWS2008/zib03.mps.gz
!gunzip datasets/linear_programming/zib03.mps.gz

And this larger instance can now be solved on 1, 2, 4 and 8 GPUs with the parameter `--num_gpus`

In [ ]:
t_zib03_1_gpu = run_cuopt(
    "datasets/linear_programming/zib03.mps",
    num_gpus=1,
)
t_zib03_1_gpu

In [ ]:
t_zib03_2gpu = run_cuopt(
    "datasets/linear_programming/zib03.mps",
    num_gpus=2,
)
t_zib03_4gpu = run_cuopt(
    "datasets/linear_programming/zib03.mps",
    num_gpus=4,
)
t_zib03_8gpu = run_cuopt(
    "datasets/linear_programming/zib03.mps",
    num_gpus=8,
)

And now let's take a look at the solve times and compare the PDLP solver performance over multiple GPUs.

In [ ]:
print(f"1 GPU : {t_zib03_1_gpu}s")
print(f"2 GPUs: {t_zib03_2gpu}s")
print(f"4 GPUs: {t_zib03_4gpu}s")
print(f"8 GPUs: {t_zib03_8gpu}s")

In [ ]:
import matplotlib.pyplot as plt

def plot_gpu_times(instance):
    instance_name = f"{instance}.mps"
    # Convert None to 1 GPU
    for r in solve_times:
        if r["num_gpus"] is None:
            r["num_gpus"] = 1

    # Filter and sort
    rows = sorted(
        [r for r in solve_times if r["instance"] == instance_name],
        key=lambda r: r["num_gpus"]
    )

    if not rows:
        print(f"No data found for {instance_name}")
        return

    gpus = [r["num_gpus"] for r in rows]
    times = [r["time_s"] for r in rows]

    plt.figure(figsize=(8, 5))
    plt.title(f"PDLP Solve Time for {instance}")
    bars = plt.bar(
        range(len(gpus)),
        times,
        color="#76B900",
        width=0.5
    )

    plt.xlabel("Number of GPUs")
    plt.ylabel("Solve time (s)")
    plt.xticks(range(len(gpus)), gpus)

    for bar, time in zip(bars, times):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            time,
            f"{time:.2f}s",
            ha="center",
            va="bottom",
        )

    plt.show()

In [ ]:
plot_gpu_times("zib03")

We encourage you to test multi-GPU PDLP on your own LPs and share with us your experience!

In [ ]:
path = "path/to/your/lp/file.mps"
run_cuopt(path, num_gpus=1)
run_cuopt(path, num_gpus=8)

plot_gpu_times(path.rsplit("/", 1)[-1])